# Comparision of different Optimizers

In this notebook, we'll compare the Tensorflow implementations of SGD, SGD + momentum, RMSprop and ADAM optimizers using the default learning rate on FashionMNIST dataset using the LeNet5 model. This is done to get a better understanding of each optimizer and how to select one based on the loss graphs. 

In [7]:
import os
import random
import numpy as np
from typing import Tuple
from dataclasses import dataclass

import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

import tensorflow as tf
from keras import Model
from keras.optimizers import SGD, RMSprop, Adam
from keras.layers import Dense, Conv2D, Activation, Flatten, MaxPool2D
from keras.utils import to_categorical

### System Configuration

In [3]:
def system_config(seed_value: int):
    np.random.seed(seed=seed_value)
    random.seed(seed_value)
    tf.random.set_seed(seed=seed_value)

    gpu_devices = tf.config.list_physical_devices("GPU")
    print("Number of GPU devices: ", len(gpu_devices))

    if len(gpu_devices) == 0:
        print("Using CPU")
        return

    print("Using GPU")
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

    # If there are any gpu devices, use first gpu.
    tf.config.experimental.set_visible_devices(gpu_devices[0], "GPU")

    # Grow the memory usage as it is needed by the process.
    tf.config.experimental.set_memory_growth(gpu_devices[0], True)

    # Enable using cudNN.
    os.environ["TF_USE_CUDNN"] = "true"

In [4]:
system_config(seed_value=7)

Number of GPU devices:  0
Using CPU


### Training and Dataset Configs

In [5]:
@dataclass(frozen=True)
class TrainingConfig:
    IMG_HEIGHT: int = 28
    IMG_WIDTH: int = 28
    IMG_CHANNELS: int = 3
    NUM_CLASSES: int = 10


@dataclass(frozen=True)
class DatasetConfig:
    EPOCHS: int = 101
    BATCH_SIZE: int = 64
    LEARNING_RATE: float = 0.001
    ROOT_LOGS_DIR = "logs"
    ROOT_CHECKPOINTS_DIR = "checkpoints"

### Get the data and visualize the samples

In [8]:
def get_data() -> Tuple[Tuple, Tuple]:
    import keras

    dataset = keras.datasets.fashion_mnist

    return dataset.load_data()

In [ ]:
def preprocess_dataset(
    train_set: Tuple,
    valid_set: Tuple,
    num_classes: int = 10,
    seed: int = 3,
    resize_to=None,
):
    (X_train, y_train) = train_set
    (X_valid, y_valid) = valid_set

    # Add axis to gray scale
    if len(X_train) != 4:
        X_train = tf.exand_dims(X_train, axis=3)
        X_valid = tf.exand_dims(X_valid, axis=3)

    print("Training shape: {X_train.shape}")
    print("Validation shape: {X_valid.shape}")
    print("Img shape: {X_train[0].shape}")

    assert num_classes == len(
        np.unique(y_train)
    ), "The number of classes in the dataset does not match the expected number of classes"

    print("Number of classes: ", num_classes)

    # One-hot encode the labels, if necessary
    if len(y_train) != 2:
        y_train = to_categorical(y_train, num_classes)